# Retail Reporting Automation: A Financial Case Study

## Research question
Is automating monthly Excel reporting financially
worthwhile when productive demand for saved employee
time is uncertain?

## Workflow
1. Generate synthetic retail reports
2. Consolidate and validate data with Python
3. Measure processing performance
4. Estimate implementation and maintenance costs
5. Evaluate investment economics using NPV
6. Visualize break-even conditions

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

PROJECT_DIR = Path(
    "/content/drive/MyDrive/"
    "Data_Analytics_Portfolio/"
    "02_Python_Automation/"
    "automation_investment_analysis"
)

INPUT_DIR = PROJECT_DIR / "input_reports"
OUTPUT_DIR = PROJECT_DIR / "output_reports"

INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project folder:", PROJECT_DIR)

In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path

# Correct project location in Google Drive
PROJECT_DIR = Path(
    "/content/drive/MyDrive/"
    "Data_Analytics_Portfolio/"
    "02_Python_Automation/"
    "automation_investment_analysis"
)

INPUT_DIR = PROJECT_DIR / "input_reports"
OUTPUT_DIR = PROJECT_DIR / "output_reports"

INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Reproducible synthetic data
rng = np.random.default_rng(42)

stores = [f"STORE_{i:02d}" for i in range(1, 9)]

products = {
    "Electronics": ["Headphones", "Keyboard", "Mouse"],
    "Home": ["Lamp", "Storage Box", "Desk Organizer"],
    "Office": ["Notebook", "Pen Set", "Paper"],
    "Clothing": ["T-Shirt", "Sweater", "Jacket"],
    "Accessories": ["Backpack", "Wallet", "Umbrella"]
}

all_reports = []

for store in stores:
    records = []

    for i in range(250):
        category = rng.choice(list(products.keys()))
        product = rng.choice(products[category])

        quantity = int(rng.integers(1, 8))
        unit_price = round(float(rng.uniform(8, 150)), 2)

        records.append({
            "Transaction_ID": f"{store}_{i + 1:04d}",
            "Date": (
                pd.Timestamp("2026-09-01")
                + pd.Timedelta(
                    days=int(rng.integers(0, 30))
                )
            ),
            "Store_ID": store,
            "Category": category,
            "Product": product,
            "Quantity": quantity,
            "Unit_Price": unit_price,
            "Revenue": round(quantity * unit_price, 2)
        })

    df_store = pd.DataFrame(records)

    file_path = INPUT_DIR / f"{store}_September_2026.xlsx"

    df_store.to_excel(
        file_path,
        index=False,
        sheet_name="Sales"
    )

    all_reports.append(df_store)

print(f"Created {len(all_reports)} Excel reports")
print(f"Total transactions: {sum(len(x) for x in all_reports)}")

display(all_reports[0].head())


In [ ]:

files = sorted(INPUT_DIR.glob("*.xlsx"))

print("Excel reports found:\n")

for file in files:
    print(file.name)

print(f"\nTotal files: {len(files)}")

assert len(files) == 8, (
    f"Expected 8 files, found {len(files)}"
)

print("\nValidation passed!")


In [ ]:

validation_results = []

required_columns = [
    "Transaction_ID",
    "Date",
    "Store_ID",
    "Category",
    "Product",
    "Quantity",
    "Unit_Price",
    "Revenue"
]

for file in files:
    df = pd.read_excel(file)

    missing_columns = [
        col for col in required_columns
        if col not in df.columns
    ]

    duplicate_ids = (
        df["Transaction_ID"].duplicated().sum()
        if not missing_columns else None
    )

    missing_values = (
        df.isna().sum().sum()
        if not missing_columns else None
    )

    validation_results.append({
        "File": file.name,
        "Rows": len(df),
        "Missing_Columns": len(missing_columns),
        "Duplicate_IDs": duplicate_ids,
        "Missing_Values": missing_values
    })

validation_df = pd.DataFrame(validation_results)

display(validation_df)


In [ ]:

validation_file = OUTPUT_DIR / "data_quality_check.xlsx"

validation_df.to_excel(
    validation_file,
    index=False,
    sheet_name="Validation"
)

print("Validation report saved to:")
print(validation_file)


In [ ]:

import pandas as pd
from pathlib import Path

# We use the existing INPUT_DIR and OUTPUT_DIR
files = sorted(INPUT_DIR.glob("STORE_*.xlsx"))

assert len(files) == 8, (
    f"Expected 8 store reports, found {len(files)}"
)

dataframes = []

for file in files:
    df = pd.read_excel(file, sheet_name="Sales")

    # Track the original source file
    df["Source_File"] = file.name

    dataframes.append(df)

# Combine all store reports
sales_df = pd.concat(
    dataframes,
    ignore_index=True
)

print("Files processed:", len(files))
print("Total transactions:", len(sales_df))
print("Total columns:", len(sales_df.columns))

display(sales_df.head())


In [ ]:

import numpy as np

# Recalculate revenue from source values
expected_revenue = (
    sales_df["Quantity"] * sales_df["Unit_Price"]
).round(2)

revenue_errors = (
    ~np.isclose(
        sales_df["Revenue"],
        expected_revenue,
        atol=0.001
    )
).sum()

duplicate_transactions = (
    sales_df["Transaction_ID"].duplicated().sum()
)

missing_values = sales_df.isna().sum().sum()

invalid_quantities = (
    sales_df["Quantity"] <= 0
).sum()

invalid_prices = (
    sales_df["Unit_Price"] < 0
).sum()

quality_checks = pd.DataFrame({
    "Check": [
        "Duplicate transaction IDs",
        "Missing values",
        "Incorrect revenue calculations",
        "Invalid quantities",
        "Negative unit prices"
    ],
    "Issues_Found": [
        duplicate_transactions,
        missing_values,
        revenue_errors,
        invalid_quantities,
        invalid_prices
    ]
})

quality_checks["Status"] = np.where(
    quality_checks["Issues_Found"] == 0,
    "PASS",
    "FAIL"
)

display(quality_checks)

assert (quality_checks["Status"] == "PASS").all(), (
    "Data quality checks failed. Review the input files."
)

print("All data quality checks passed!")


In [ ]:

# Total revenue
total_revenue = sales_df["Revenue"].sum()

# Total units sold
total_units = sales_df["Quantity"].sum()

# Average transaction value
average_transaction = sales_df["Revenue"].mean()

# Store performance
store_summary = (
    sales_df.groupby("Store_ID")
    .agg(
        Transactions=("Transaction_ID", "count"),
        Units_Sold=("Quantity", "sum"),
        Total_Revenue=("Revenue", "sum")
    )
    .reset_index()
    .sort_values("Total_Revenue", ascending=False)
)

# Category performance
category_summary = (
    sales_df.groupby("Category")
    .agg(
        Transactions=("Transaction_ID", "count"),
        Units_Sold=("Quantity", "sum"),
        Total_Revenue=("Revenue", "sum")
    )
    .reset_index()
    .sort_values("Total_Revenue", ascending=False)
)

# Revenue share per store
store_summary["Revenue_Share_Pct"] = (
    store_summary["Total_Revenue"]
    / total_revenue * 100
)

# Revenue share per category
category_summary["Revenue_Share_Pct"] = (
    category_summary["Total_Revenue"]
    / total_revenue * 100
)

print(f"Total revenue: €{total_revenue:,.2f}")
print(f"Units sold: {total_units:,}")
print(f"Average transaction: €{average_transaction:,.2f}")

print("\nStore performance:")
display(store_summary)

print("\nCategory performance:")
display(category_summary)


In [ ]:

from openpyxl.styles import (
    Font, PatternFill, Alignment
)
from openpyxl.utils import get_column_letter

report_path = OUTPUT_DIR / "monthly_retail_report.xlsx"

executive_summary = pd.DataFrame({
    "Metric": [
        "Reporting Month",
        "Number of Stores",
        "Total Transactions",
        "Total Units Sold",
        "Total Revenue (EUR)",
        "Average Transaction Value (EUR)"
    ],
    "Value": [
        "September 2026",
        sales_df["Store_ID"].nunique(),
        len(sales_df),
        total_units,
        total_revenue,
        average_transaction
    ]
})

with pd.ExcelWriter(
    report_path,
    engine="openpyxl"
) as writer:

    executive_summary.to_excel(
        writer,
        sheet_name="Executive Summary",
        index=False
    )

    store_summary.to_excel(
        writer,
        sheet_name="Store Performance",
        index=False
    )

    category_summary.to_excel(
        writer,
        sheet_name="Category Performance",
        index=False
    )

    sales_df.to_excel(
        writer,
        sheet_name="All Transactions",
        index=False
    )

    quality_checks.to_excel(
        writer,
        sheet_name="Data Quality",
        index=False
    )

    workbook = writer.book

    for sheet in workbook.worksheets:

        # Header styling
        for cell in sheet[1]:
            cell.fill = PatternFill(
                start_color="17365D",
                end_color="17365D",
                fill_type="solid"
            )
            cell.font = Font(
                color="FFFFFF",
                bold=True
            )
            cell.alignment = Alignment(
                horizontal="center"
            )

        # Freeze the header
        sheet.freeze_panes = "A2"

        # Enable filters
        sheet.auto_filter.ref = sheet.dimensions

        # Set column widths
        for column_cells in sheet.columns:
            max_length = max(
                len(str(cell.value))
                if cell.value is not None else 0
                for cell in column_cells
            )

            column_letter = get_column_letter(
                column_cells[0].column
            )

            sheet.column_dimensions[
                column_letter
            ].width = min(max_length + 3, 42)

        # Format numeric values in summary sheets
        if sheet.title in [
            "Store Performance",
            "Category Performance"
        ]:
            headers = {
                cell.value: cell.column
                for cell in sheet[1]
            }

            for row in sheet.iter_rows(min_row=2):
                row[
                    headers["Total_Revenue"] - 1
                ].number_format = '#,##0.00 "€"'

                row[
                    headers["Revenue_Share_Pct"] - 1
                ].number_format = '0.0"%"'

        if sheet.title == "Executive Summary":
            sheet["B6"].number_format = '#,##0.00 "€"'
            sheet["B7"].number_format = '#,##0.00 "€"'

print("Report created successfully:")
print(report_path)


In [ ]:

# Read the saved report
saved_transactions = pd.read_excel(
    report_path,
    sheet_name="All Transactions"
)

saved_stores = pd.read_excel(
    report_path,
    sheet_name="Store Performance"
)

saved_categories = pd.read_excel(
    report_path,
    sheet_name="Category Performance"
)

# Reconcile all totals
assert len(saved_transactions) == 2000

assert np.isclose(
    saved_transactions["Revenue"].sum(),
    total_revenue
)

assert np.isclose(
    saved_stores["Total_Revenue"].sum(),
    total_revenue
)

assert np.isclose(
    saved_categories["Total_Revenue"].sum(),
    total_revenue
)

print("Reconciliation successful!")
print("All revenue totals match.")


In [ ]:
# Compare revenue totals across all report sheets

transactions = pd.read_excel(
    report_path, sheet_name="All Transactions"
)

stores = pd.read_excel(
    report_path, sheet_name="Store Performance"
)

categories = pd.read_excel(
    report_path, sheet_name="Category Performance"
)

summary = pd.read_excel(
    report_path, sheet_name="Executive Summary"
)

# Read the total revenue from Executive Summary
summary_revenue = summary.loc[
    summary["Metric"] == "Total Revenue (EUR)",
    "Value"
].iloc[0]

# Compare all totals, rounded to cents
reconciliation = pd.DataFrame({
    "Source": [
        "All Transactions",
        "Store Performance",
        "Category Performance",
        "Executive Summary"
    ],
    "Total Revenue (EUR)": [
        round(transactions["Revenue"].sum(), 2),
        round(stores["Total_Revenue"].sum(), 2),
        round(categories["Total_Revenue"].sum(), 2),
        round(float(summary_revenue), 2)
    ]
})

reference = reconciliation.loc[
    0, "Total Revenue (EUR)"
]

reconciliation["Difference (EUR)"] = (
    reconciliation["Total Revenue (EUR)"] - reference
).round(2)

reconciliation["Status"] = reconciliation[
    "Difference (EUR)"
].apply(lambda x: "PASS" if x == 0 else "FAIL")

display(reconciliation)

In [ ]:

import time
import pandas as pd
import numpy as np

from pathlib import Path

def run_reporting_pipeline():

    files = sorted(INPUT_DIR.glob("STORE_*.xlsx"))

    if len(files) != 8:
        raise ValueError("Expected exactly 8 Excel files")

    # 1. Read and consolidate
    frames = []

    for file in files:
        df = pd.read_excel(file, sheet_name="Sales")
        df["Source_File"] = file.name
        frames.append(df)

    sales = pd.concat(frames, ignore_index=True)

    # 2. Validate
    expected_revenue = (
        sales["Quantity"] * sales["Unit_Price"]
    ).round(2)

    assert len(sales) == 2000
    assert not sales["Transaction_ID"].duplicated().any()
    assert not sales.isna().any().any()
    assert (sales["Quantity"] > 0).all()
    assert (sales["Unit_Price"] >= 0).all()
    assert np.allclose(
        sales["Revenue"], expected_revenue, atol=0.001
    )

    # 3. Calculate business metrics
    total_revenue = sales["Revenue"].sum()

    stores = (
        sales.groupby("Store_ID")
        .agg(
            Transactions=("Transaction_ID", "count"),
            Units_Sold=("Quantity", "sum"),
            Total_Revenue=("Revenue", "sum")
        )
        .reset_index()
    )

    categories = (
        sales.groupby("Category")
        .agg(
            Transactions=("Transaction_ID", "count"),
            Units_Sold=("Quantity", "sum"),
            Total_Revenue=("Revenue", "sum")
        )
        .reset_index()
    )

    stores["Revenue_Share_Pct"] = (
        stores["Total_Revenue"] / total_revenue * 100
    )

    categories["Revenue_Share_Pct"] = (
        categories["Total_Revenue"] / total_revenue * 100
    )

    # 4. Create Excel report
    report_file = (
        OUTPUT_DIR / "monthly_retail_report_benchmark.xlsx"
    )

    with pd.ExcelWriter(
        report_file, engine="openpyxl"
    ) as writer:

        sales.to_excel(
            writer,
            sheet_name="All Transactions",
            index=False
        )

        stores.to_excel(
            writer,
            sheet_name="Store Performance",
            index=False
        )

        categories.to_excel(
            writer,
            sheet_name="Category Performance",
            index=False
        )

        summary = pd.DataFrame({
            "Metric": [
                "Total Revenue (EUR)",
                "Total Transactions",
                "Total Units Sold",
                "Average Transaction Value (EUR)"
            ],
            "Value": [
                total_revenue,
                len(sales),
                sales["Quantity"].sum(),
                sales["Revenue"].mean()
            ]
        })

        summary.to_excel(
            writer,
            sheet_name="Executive Summary",
            index=False
        )

        checks = pd.DataFrame({
            "Check": [
                "Duplicate IDs",
                "Missing values",
                "Revenue mismatch"
            ],
            "Issues": [0, 0, 0]
        })

        checks.to_excel(
            writer,
            sheet_name="Data Quality",
            index=False
        )

    # 5. Reconciliation
    assert np.isclose(
        stores["Total_Revenue"].sum(),
        total_revenue
    )

    assert np.isclose(
        categories["Total_Revenue"].sum(),
        total_revenue
    )

    return report_file


In [ ]:

measurements = []

for run in range(1, 6):

    start = time.perf_counter()

    report_file = run_reporting_pipeline()

    elapsed_seconds = time.perf_counter() - start

    measurements.append({
        "Run": run,
        "Processing_Seconds": elapsed_seconds
    })

benchmark_df = pd.DataFrame(measurements)

display(benchmark_df)

print(
    "Median processing time:",
    round(
        benchmark_df["Processing_Seconds"].median(),
        3
    ),
    "seconds"
)


In [ ]:

manual_tasks = pd.DataFrame({
    "Task": [
        "Open and inspect files",
        "Copy and consolidate data",
        "Create summary tables",
        "Calculate KPIs",
        "Validate and save report"
    ],
    "Units": [8, 8, 2, 4, 1],
    "Minutes_Per_Unit": [1.5, 2, 5, 1, 8]
})

manual_tasks["Total_Minutes"] = (
    manual_tasks["Units"]
    * manual_tasks["Minutes_Per_Unit"]
)

manual_estimate = manual_tasks[
    "Total_Minutes"
].sum()

display(manual_tasks)

print(
    f"Estimated manual effort: "
    f"{manual_estimate:.1f} minutes"
)


In [ ]:

automated_tasks = pd.DataFrame({
    "Task": [
        "Prepare input files",
        "Start and monitor workflow",
        "Review final report"
    ],
    "Minutes": [2, 1, 4]
})

automated_human_effort = (
    automated_tasks["Minutes"].sum()
)

processing_minutes = (
    benchmark_df["Processing_Seconds"].median() / 60
)

estimated_labor_savings = (
    manual_estimate - automated_human_effort
)

print(
    f"Estimated manual effort: "
    f"{manual_estimate:.1f} min"
)

print(
    f"Estimated automated human effort: "
    f"{automated_human_effort:.1f} min"
)

print(
    f"Measured Python processing: "
    f"{processing_minutes:.3f} min"
)

print(
    f"Estimated labor time saved: "
    f"{estimated_labor_savings:.1f} min"
)

print(
    f"Estimated reduction in labor effort: "
    f"{estimated_labor_savings / manual_estimate:.1%}"
)


In [ ]:

MEASUREMENT_DIR = PROJECT_DIR / "measurements"
MEASUREMENT_DIR.mkdir(parents=True, exist_ok=True)

measurement_file = (
    MEASUREMENT_DIR / "automation_efficiency_study.xlsx"
)

comparison = pd.DataFrame({
    "Metric": [
        "Manual human effort",
        "Automated human effort",
        "Python processing time",
        "Estimated labor time saved",
        "Estimated labor reduction"
    ],
    "Value": [
        manual_estimate,
        automated_human_effort,
        processing_minutes,
        estimated_labor_savings,
        estimated_labor_savings / manual_estimate
    ],
    "Unit": [
        "minutes",
        "minutes",
        "minutes",
        "minutes",
        "fraction"
    ],
    "Evidence_Type": [
        "Scenario assumption",
        "Scenario assumption",
        "Measured",
        "Calculated from assumptions",
        "Calculated from assumptions"
    ]
})

with pd.ExcelWriter(
    measurement_file,
    engine="openpyxl"
) as writer:

    benchmark_df.to_excel(
        writer,
        sheet_name="Python Benchmark",
        index=False
    )

    manual_tasks.to_excel(
        writer,
        sheet_name="Manual Estimate",
        index=False
    )

    automated_tasks.to_excel(
        writer,
        sheet_name="Automated Human Effort",
        index=False
    )

    comparison.to_excel(
        writer,
        sheet_name="Comparison",
        index=False
    )

print("Study saved to:")
print(measurement_file)


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

MEASUREMENT_FILE = (
    PROJECT_DIR
    / "measurements"
    / "automation_efficiency_study.xlsx"
)

comparison = pd.read_excel(
    MEASUREMENT_FILE,
    sheet_name="Comparison"
)

display(comparison)

def get_metric(metric_name):
    result = comparison.loc[
        comparison["Metric"] == metric_name,
        "Value"
    ]

    if len(result) != 1:
        raise ValueError(
            f"Metric not found or duplicated: {metric_name}"
        )

    return float(result.iloc[0])

manual_minutes = get_metric("Manual human effort")
automated_minutes = get_metric("Automated human effort")

print(f"Manual estimate: {manual_minutes:.1f} min")
print(f"Automated estimate: {automated_minutes:.1f} min")


In [ ]:

# Illustrative assumptions — NOT observed costs

cost_assumptions = {
    "Development hours": 24,
    "Testing and deployment hours": 8,
    "Training hours": 4,
    "Implementation cost per hour (EUR)": 45,
    "Annual maintenance hours": 12,
    "Maintenance cost per hour (EUR)": 45,
    "Annual software cost (EUR)": 0
}

initial_investment = (
    (
        cost_assumptions["Development hours"]
        + cost_assumptions["Testing and deployment hours"]
        + cost_assumptions["Training hours"]
    )
    * cost_assumptions["Implementation cost per hour (EUR)"]
)

annual_maintenance = (
    cost_assumptions["Annual maintenance hours"]
    * cost_assumptions["Maintenance cost per hour (EUR)"]
    + cost_assumptions["Annual software cost (EUR)"]
)

print(f"Illustrative investment: €{initial_investment:,.0f}")
print(f"Illustrative annual maintenance: €{annual_maintenance:,.0f}")


In [ ]:

def automation_npv(
    reports_per_month,
    manual_minutes_per_report,
    automated_minutes_per_report,
    available_demand_hours,
    contribution_per_hour,
    initial_cost,
    annual_cost,
    discount_rate,
    project_years
):
    # Labor capacity released by automation
    saved_minutes = max(
        0,
        manual_minutes_per_report
        - automated_minutes_per_report
    )

    released_hours = (
        reports_per_month * saved_minutes / 60
    )

    # Business demand limits the realizable benefit
    productive_hours = min(
        released_hours,
        available_demand_hours
    )

    # Incremental cash contribution, not payroll savings
    annual_benefit = (
        productive_hours
        * contribution_per_hour
        * 12
    )

    annual_net_cash_flow = (
        annual_benefit - annual_cost
    )

    npv = -initial_cost + sum(
        annual_net_cash_flow
        / (1 + discount_rate) ** year
        for year in range(1, project_years + 1)
    )

    return {
        "NPV": npv,
        "Released_Hours_Per_Month": released_hours,
        "Productive_Hours_Per_Month": productive_hours,
        "Unused_Hours_Per_Month": (
            released_hours - productive_hours
        ),
        "Annual_Benefit": annual_benefit
    }


In [ ]:

# Illustrative scenario settings
reports_per_month = 20
available_demand_hours = 8
contribution_per_hour = 30
discount_rate = 0.10
project_years = 3

scenarios = {
    "Low manual effort": 30,
    "Reference estimate": manual_minutes,
    "High manual effort": 70
}

results = []

for name, manual_time in scenarios.items():

    result = automation_npv(
        reports_per_month=reports_per_month,
        manual_minutes_per_report=manual_time,
        automated_minutes_per_report=automated_minutes,
        available_demand_hours=available_demand_hours,
        contribution_per_hour=contribution_per_hour,
        initial_cost=initial_investment,
        annual_cost=annual_maintenance,
        discount_rate=discount_rate,
        project_years=project_years
    )

    results.append({
        "Scenario": name,
        "Manual_Minutes": manual_time,
        **result
    })

scenario_df = pd.DataFrame(results)

display(scenario_df.round(2))


In [ ]:

from matplotlib.colors import TwoSlopeNorm

report_range = np.arange(1, 61)
demand_range = np.linspace(0, 30, 121)

npv_values = np.zeros((
    len(demand_range),
    len(report_range)
))

for i, demand in enumerate(demand_range):
    for j, report_count in enumerate(report_range):

        result = automation_npv(
            reports_per_month=report_count,
            manual_minutes_per_report=manual_minutes,
            automated_minutes_per_report=automated_minutes,
            available_demand_hours=demand,
            contribution_per_hour=contribution_per_hour,
            initial_cost=initial_investment,
            annual_cost=annual_maintenance,
            discount_rate=discount_rate,
            project_years=project_years
        )

        npv_values[i, j] = result["NPV"]

X, Y = np.meshgrid(report_range, demand_range)

fig, ax = plt.subplots(figsize=(11, 7))

max_abs = np.max(np.abs(npv_values))

heatmap = ax.pcolormesh(
    X,
    Y,
    npv_values,
    cmap="RdYlGn",
    norm=TwoSlopeNorm(
        vmin=-max_abs,
        vcenter=0,
        vmax=max_abs
    ),
    shading="auto"
)

# Break-even contour
if npv_values.min() < 0 < npv_values.max():
    contour = ax.contour(
        X, Y, npv_values,
        levels=[0],
        colors="black",
        linewidths=2.5
    )

    ax.clabel(
        contour,
        fmt={0: "NPV = 0"},
        fontsize=10
    )

# Reference scenario
ax.scatter(
    reports_per_month,
    available_demand_hours,
    color="black",
    edgecolors="white",
    s=150,
    marker="D",
    zorder=5,
    label="Illustrative scenario"
)

ax.set_title(
    "When Does Reporting Automation Pay Off?",
    fontsize=17,
    fontweight="bold",
    pad=20
)

ax.set_xlabel("Reports automated per month")
ax.set_ylabel("Available productive demand (hours/month)")

colorbar = fig.colorbar(heatmap, ax=ax)
colorbar.set_label("3-Year NPV (EUR)")

ax.legend(loc="upper left")

fig.tight_layout()
plt.show()


In [ ]:

MODEL_DIR = PROJECT_DIR / "financial_model"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

model_file = (
    MODEL_DIR / "automation_financial_model.xlsx"
)

assumptions_df = pd.DataFrame([
    ["Manual effort", manual_minutes, "min/report", "Scenario estimate"],
    ["Automated human effort", automated_minutes, "min/report", "Scenario estimate"],
    ["Reports per month", reports_per_month, "reports", "Scenario assumption"],
    ["Available demand", available_demand_hours, "hours/month", "Scenario assumption"],
    ["Contribution per hour", contribution_per_hour, "EUR/hour", "Scenario assumption"],
    ["Initial investment", initial_investment, "EUR", "Cost-model estimate"],
    ["Annual maintenance", annual_maintenance, "EUR/year", "Cost-model estimate"],
    ["Discount rate", discount_rate, "fraction", "Scenario assumption"],
    ["Project lifetime", project_years, "years", "Scenario assumption"]
], columns=[
    "Parameter", "Value", "Unit", "Evidence_Type"
])

heatmap_df = pd.DataFrame(
    npv_values,
    index=demand_range,
    columns=report_range
)

heatmap_df.index.name = "Demand_Hours_Per_Month"
heatmap_df.columns.name = "Reports_Per_Month"

with pd.ExcelWriter(
    model_file,
    engine="openpyxl"
) as writer:

    assumptions_df.to_excel(
        writer,
        sheet_name="Assumptions",
        index=False
    )

    pd.DataFrame(
        list(cost_assumptions.items()),
        columns=["Cost Component", "Value"]
    ).to_excel(
        writer,
        sheet_name="Cost Breakdown",
        index=False
    )

    scenario_df.to_excel(
        writer,
        sheet_name="Scenarios",
        index=False
    )

    heatmap_df.to_excel(
        writer,
        sheet_name="NPV Sensitivity"
    )

figure_path = MODEL_DIR / "automation_npv_heatmap.png"

fig.savefig(
    figure_path,
    dpi=220,
    bbox_inches="tight"
)

print("Financial model:", model_file)
print("Heatmap:", figure_path)


In [ ]:

# Step 23 — Break-even analysis

annuity_factor = sum(
    1 / (1 + discount_rate) ** year
    for year in range(1, project_years + 1)
)

required_annual_benefit = (
    initial_investment / annuity_factor
    + annual_maintenance
)

required_monthly_benefit = (
    required_annual_benefit / 12
)

print(
    f"Required monthly cash benefit: "
    f"€{required_monthly_benefit:.2f}"
)

if contribution_per_hour > 0:
    required_productive_hours = (
        required_monthly_benefit
        / contribution_per_hour
    )

    print(
        f"Required productive hours at "
        f"€{contribution_per_hour:.0f}/hour: "
        f"{required_productive_hours:.2f}"
    )


In [ ]:

# Step 24 — Investment decision thresholds

def minimum_contribution_per_hour(
    reports_per_month,
    manual_minutes,
    automated_minutes,
    demand_hours
):
    released_hours = (
        reports_per_month
        * max(manual_minutes - automated_minutes, 0)
        / 60
    )

    productive_hours = min(
        released_hours,
        demand_hours
    )

    if productive_hours <= 0:
        return np.inf

    return (
        required_monthly_benefit
        / productive_hours
    )


def maximum_initial_investment(
    reports_per_month,
    manual_minutes,
    automated_minutes,
    demand_hours,
    contribution_per_hour
):
    released_hours = (
        reports_per_month
        * max(manual_minutes - automated_minutes, 0)
        / 60
    )

    productive_hours = min(
        released_hours,
        demand_hours
    )

    annual_net_benefit = (
        12 * productive_hours * contribution_per_hour
        - annual_maintenance
    )

    return annual_net_benefit * annuity_factor


threshold_results = pd.DataFrame({
    "Metric": [
        "Required monthly cash benefit",
        "Minimum contribution per productive hour",
        "Maximum initial investment"
    ],
    "Value": [
        required_monthly_benefit,
        minimum_contribution_per_hour(
            reports_per_month,
            manual_minutes,
            automated_minutes,
            available_demand_hours
        ),
        maximum_initial_investment(
            reports_per_month,
            manual_minutes,
            automated_minutes,
            available_demand_hours,
            contribution_per_hour
        )
    ],
    "Unit": [
        "EUR/month",
        "EUR/productive hour",
        "EUR"
    ]
})

display(threshold_results.round(2))


In [ ]:

import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

# Three illustrative scales
scale_scenarios = [1, 5, 20]

# X = demand for economically useful work
demand_values = np.linspace(0, 25, 151)

# Y = incremental cash contribution per productive hour
contribution_values = np.linspace(0, 80, 161)

D, C = np.meshgrid(
    demand_values,
    contribution_values
)

npv_maps = []

for scale in scale_scenarios:
    released_hours = (
        scale
        * max(manual_minutes - automated_minutes, 0)
        / 60
    )

    productive_hours = np.minimum(
        D,
        released_hours
    )

    annual_benefit = (
        12 * productive_hours * C
    )

    npv_map = (
        -initial_investment
        + (annual_benefit - annual_maintenance)
        * annuity_factor
    )

    npv_maps.append(npv_map)

# One common color scale for honest comparison
max_abs = max(
    np.max(np.abs(m))
    for m in npv_maps
)

norm = TwoSlopeNorm(
    vmin=-max_abs,
    vcenter=0,
    vmax=max_abs
)

fig, axes = plt.subplots(
    1, 3,
    figsize=(16, 5.8),
    sharex=True,
    sharey=True,
    constrained_layout=True
)

for ax, scale, npv_map in zip(
    axes, scale_scenarios, npv_maps
):
    im = ax.pcolormesh(
        D,
        C,
        npv_map,
        cmap="RdYlGn",
        norm=norm,
        shading="auto"
    )

    # NPV = 0 break-even boundary
    if npv_map.min() < 0 < npv_map.max():
        ax.contour(
            D,
            C,
            npv_map,
            levels=[0],
            colors="black",
            linewidths=2
        )

    released_hours = (
        scale
        * max(manual_minutes - automated_minutes, 0)
        / 60
    )

    # Demand saturation threshold
    if released_hours <= demand_values.max():
        ax.axvline(
            released_hours,
            color="#303030",
            linestyle=":",
            linewidth=1.5,
            alpha=0.8
        )

    ax.set_title(
        f"{scale} reports/month\n"
        f"{released_hours:.1f} h released",
        fontsize=12,
        fontweight="bold"
    )

    ax.set_xlabel(
        "Productive demand (h/month)"
    )

    ax.grid(alpha=0.08)

axes[0].set_ylabel(
    "Incremental cash contribution (€/h)"
)

fig.suptitle(
    "Automation Value Depends on What Happens After Time Is Saved",
    fontsize=17,
    fontweight="bold"
)

colorbar = fig.colorbar(
    im,
    ax=axes,
    shrink=0.85,
    pad=0.02
)

colorbar.set_label("3-year NPV (€)")

plt.show()


In [ ]:

manual_time_scenarios = [30, 40, 50, 60, 70]
report_scales = [1, 5, 10, 20, 40]

sensitivity_rows = []

for manual_time in manual_time_scenarios:
    for scale in report_scales:

        result = automation_npv(
            reports_per_month=scale,
            manual_minutes_per_report=manual_time,
            automated_minutes_per_report=automated_minutes,
            available_demand_hours=available_demand_hours,
            contribution_per_hour=contribution_per_hour,
            initial_cost=initial_investment,
            annual_cost=annual_maintenance,
            discount_rate=discount_rate,
            project_years=project_years
        )

        sensitivity_rows.append({
            "Manual_Minutes": manual_time,
            "Reports_Per_Month": scale,
            "NPV_EUR": result["NPV"],
            "Released_Hours": result[
                "Released_Hours_Per_Month"
            ],
            "Productive_Hours": result[
                "Productive_Hours_Per_Month"
            ]
        })

sensitivity_df = pd.DataFrame(
    sensitivity_rows
)

sensitivity_matrix = sensitivity_df.pivot(
    index="Manual_Minutes",
    columns="Reports_Per_Month",
    values="NPV_EUR"
)

display(sensitivity_matrix.round(0))


In [ ]:

MODEL_DIR = PROJECT_DIR / "financial_model"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

decision_file = (
    MODEL_DIR / "automation_decision_analysis.xlsx"
)

with pd.ExcelWriter(
    decision_file,
    engine="openpyxl"
) as writer:

    assumptions_df.to_excel(
        writer,
        sheet_name="Assumptions",
        index=False
    )

    threshold_results.to_excel(
        writer,
        sheet_name="Break Even",
        index=False
    )

    sensitivity_df.to_excel(
        writer,
        sheet_name="Sensitivity Detail",
        index=False
    )

    sensitivity_matrix.to_excel(
        writer,
        sheet_name="Sensitivity Matrix"
    )

    for scale, npv_map in zip(
        scale_scenarios, npv_maps
    ):
        pd.DataFrame(
            npv_map,
            index=contribution_values,
            columns=demand_values
        ).to_excel(
            writer,
            sheet_name=f"NPV {scale} Reports"
        )

chart_path = (
    MODEL_DIR / "automation_decision_map.png"
)

fig.savefig(
    chart_path,
    dpi=250,
    bbox_inches="tight",
    facecolor="white"
)

print("Analysis saved:", decision_file)
print("Chart saved:", chart_path)


In [ ]:

# STEP 28 — Scale-dependent implementation costs

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

MODEL_DIR = PROJECT_DIR / "financial_model"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Existing assumptions from previous steps:
# initial_investment = 1620 EUR (illustrative)
# annual_maintenance = 540 EUR (illustrative)
# manual_minutes = 50 (estimated)
# automated_minutes = 7 (estimated)
# discount_rate = 0.10 (illustrative)
# project_years = 3 (illustrative)

# Additional cost per report beyond the first one.
# These are sensitivity scenarios, NOT market estimates.
additional_setup_cost_scenarios = [0, 50, 150, 300]

# Additional annual maintenance cost per extra report.
additional_maintenance_scenarios = [0, 10, 30]

def scaled_costs(
    reports_per_month,
    setup_cost_per_extra_report=0,
    annual_maintenance_per_extra_report=0
):
    extra_reports = max(reports_per_month - 1, 0)

    investment = (
        initial_investment
        + extra_reports * setup_cost_per_extra_report
    )

    maintenance = (
        annual_maintenance
        + extra_reports
        * annual_maintenance_per_extra_report
    )

    return investment, maintenance


cost_rows = []

for scale in [1, 5, 10, 20, 40]:
    for extra_setup in additional_setup_cost_scenarios:

        investment, maintenance = scaled_costs(
            reports_per_month=scale,
            setup_cost_per_extra_report=extra_setup,
            annual_maintenance_per_extra_report=10
        )

        cost_rows.append({
            "Reports_Per_Month": scale,
            "Extra_Setup_Cost_EUR": extra_setup,
            "Initial_Investment_EUR": investment,
            "Annual_Maintenance_EUR": maintenance
        })

scaling_cost_df = pd.DataFrame(cost_rows)

display(scaling_cost_df)


In [ ]:

# STEP 29 — Scalable NPV model

def scalable_automation_npv(
    reports_per_month,
    manual_minutes_per_report,
    automated_minutes_per_report,
    productive_demand_hours,
    contribution_per_hour,
    extra_setup_cost=0,
    extra_annual_maintenance=0,
    fixed_investment=None,
    fixed_annual_maintenance=None,
    rate=None,
    years=None
):
    if fixed_investment is None:
        fixed_investment = initial_investment

    if fixed_annual_maintenance is None:
        fixed_annual_maintenance = annual_maintenance

    if rate is None:
        rate = discount_rate

    if years is None:
        years = project_years

    extra_reports = max(reports_per_month - 1, 0)

    total_investment = (
        fixed_investment
        + extra_reports * extra_setup_cost
    )

    total_annual_maintenance = (
        fixed_annual_maintenance
        + extra_reports * extra_annual_maintenance
    )

    saved_minutes_per_report = max(
        manual_minutes_per_report
        - automated_minutes_per_report,
        0
    )

    released_hours = (
        reports_per_month
        * saved_minutes_per_report
        / 60
    )

    productive_hours = min(
        released_hours,
        productive_demand_hours
    )

    annual_cash_benefit = (
        productive_hours
        * contribution_per_hour
        * 12
    )

    annuity_factor = sum(
        1 / (1 + rate) ** t
        for t in range(1, years + 1)
    )

    npv = (
        -total_investment
        + (
            annual_cash_benefit
            - total_annual_maintenance
        ) * annuity_factor
    )

    return {
        "NPV_EUR": npv,
        "Initial_Investment_EUR": total_investment,
        "Annual_Maintenance_EUR": total_annual_maintenance,
        "Released_Hours_Per_Month": released_hours,
        "Productive_Hours_Per_Month": productive_hours,
        "Unused_Hours_Per_Month": (
            released_hours - productive_hours
        ),
        "Annual_Cash_Benefit_EUR": annual_cash_benefit
    }


In [ ]:

# STEP 30 — Decision thresholds

TEST_EXTRA_SETUP = 50
TEST_EXTRA_MAINTENANCE = 10

annuity_factor = sum(
    1 / (1 + discount_rate) ** year
    for year in range(1, project_years + 1)
)

def investment_thresholds(
    reports_per_month,
    manual_time,
    automated_time,
    demand_hours,
    contribution_eur,
    extra_setup,
    extra_maintenance
):
    extra_reports = max(reports_per_month - 1, 0)

    investment = (
        initial_investment
        + extra_reports * extra_setup
    )

    maintenance = (
        annual_maintenance
        + extra_reports * extra_maintenance
    )

    released_hours = (
        reports_per_month
        * max(manual_time - automated_time, 0)
        / 60
    )

    productive_hours = min(
        released_hours,
        demand_hours
    )

    required_monthly_cash = (
        investment / annuity_factor
        + maintenance
    ) / 12

    minimum_contribution = (
        required_monthly_cash / productive_hours
        if productive_hours > 0
        else np.inf
    )

    minimum_demand = (
        required_monthly_cash / contribution_eur
        if contribution_eur > 0
        else np.inf
    )

    # Break-even may be impossible at this scale
    demand_feasible = (
        minimum_demand <= released_hours
    )

    max_investment = (
        (
            12 * productive_hours * contribution_eur
            - maintenance
        )
        * annuity_factor
    )

    return {
        "Reports_Per_Month": reports_per_month,
        "Released_Hours": released_hours,
        "Required_Monthly_Cash_EUR": required_monthly_cash,
        "Minimum_Contribution_EUR_H": minimum_contribution,
        "Minimum_Demand_Hours": minimum_demand,
        "Demand_Threshold_Feasible": demand_feasible,
        "Maximum_Initial_Investment_EUR": max_investment
    }


threshold_rows = []

for scale in [1, 5, 10, 20, 40]:
    threshold_rows.append(
        investment_thresholds(
            reports_per_month=scale,
            manual_time=manual_minutes,
            automated_time=automated_minutes,
            demand_hours=available_demand_hours,
            contribution_eur=contribution_per_hour,
            extra_setup=TEST_EXTRA_SETUP,
            extra_maintenance=TEST_EXTRA_MAINTENANCE
        )
    )

scalable_threshold_df = pd.DataFrame(
    threshold_rows
)

display(scalable_threshold_df.round(2))


In [ ]:

# STEP 31 — Multi-factor sensitivity analysis

manual_time_range = [30, 40, 50, 60, 70]
scale_range = [1, 5, 10, 20, 40]
setup_cost_range = [0, 50, 150, 300]

stress_results = []

for manual_time in manual_time_range:
    for scale in scale_range:
        for setup_cost in setup_cost_range:

            result = scalable_automation_npv(
                reports_per_month=scale,
                manual_minutes_per_report=manual_time,
                automated_minutes_per_report=automated_minutes,
                productive_demand_hours=available_demand_hours,
                contribution_per_hour=contribution_per_hour,
                extra_setup_cost=setup_cost,
                extra_annual_maintenance=10
            )

            stress_results.append({
                "Manual_Minutes": manual_time,
                "Reports_Per_Month": scale,
                "Extra_Setup_EUR": setup_cost,
                **result
            })

stress_df = pd.DataFrame(stress_results)

display(stress_df.head(10))

stress_summary = (
    stress_df.groupby("Extra_Setup_EUR")
    .agg(
        Minimum_NPV=("NPV_EUR", "min"),
        Median_NPV=("NPV_EUR", "median"),
        Maximum_NPV=("NPV_EUR", "max"),
        Positive_Scenarios=(
            "NPV_EUR",
            lambda x: int((x > 0).sum())
        ),
        Total_Scenarios=("NPV_EUR", "count")
    )
    .reset_index()
)

stress_summary["Positive_Share_Pct"] = (
    stress_summary["Positive_Scenarios"]
    / stress_summary["Total_Scenarios"]
    * 100
)

display(stress_summary.round(2))


In [ ]:

# STEP 32 — Two-part decision visualization

from matplotlib.colors import TwoSlopeNorm

VISUAL_SCALE = 20
VISUAL_EXTRA_SETUP = 50
VISUAL_EXTRA_MAINTENANCE = 10

demand_axis = np.linspace(0, 20, 161)
contribution_axis = np.linspace(0, 80, 161)

D, C = np.meshgrid(
    demand_axis,
    contribution_axis
)

released_at_scale = (
    VISUAL_SCALE
    * max(manual_minutes - automated_minutes, 0)
    / 60
)

visual_investment = (
    initial_investment
    + (VISUAL_SCALE - 1) * VISUAL_EXTRA_SETUP
)

visual_maintenance = (
    annual_maintenance
    + (VISUAL_SCALE - 1) * VISUAL_EXTRA_MAINTENANCE
)

npv_surface = (
    -visual_investment
    + (
        12 * np.minimum(D, released_at_scale) * C
        - visual_maintenance
    ) * annuity_factor
)

# Panel B — NPV as released capacity grows,
# holding productive demand fixed
released_axis = np.linspace(0, 25, 200)

npv_by_capacity = (
    -visual_investment
    + (
        12
        * np.minimum(
            released_axis,
            available_demand_hours
        )
        * contribution_per_hour
        - visual_maintenance
    ) * annuity_factor
)

fig_final, (ax1, ax2) = plt.subplots(
    1, 2,
    figsize=(15, 6),
    gridspec_kw={"width_ratios": [1.25, 1]},
    constrained_layout=True
)

# PANEL A
max_abs = np.max(np.abs(npv_surface))

mesh = ax1.pcolormesh(
    D,
    C,
    npv_surface,
    cmap="RdYlGn",
    norm=TwoSlopeNorm(
        vmin=-max_abs,
        vcenter=0,
        vmax=max_abs
    ),
    shading="auto"
)

if npv_surface.min() < 0 < npv_surface.max():
    ax1.contour(
        D, C, npv_surface,
        levels=[0],
        colors="black",
        linewidths=2.5
    )

ax1.axvline(
    released_at_scale,
    color="#333333",
    linestyle=":",
    linewidth=1.5
)

ax1.set_title(
    "A. When Does Automation Pay Off?",
    fontweight="bold"
)

ax1.set_xlabel("Productive demand (hours/month)")
ax1.set_ylabel("Incremental cash contribution (€/hour)")

cbar = fig_final.colorbar(
    mesh, ax=ax1, shrink=0.85
)
cbar.set_label("3-year NPV (€)")

# PANEL B
ax2.plot(
    released_axis,
    npv_by_capacity,
    color="#167c62",
    linewidth=3
)

ax2.axhline(
    0,
    color="#444444",
    linewidth=1
)

ax2.axvline(
    available_demand_hours,
    color="#555555",
    linestyle="--",
    linewidth=1.5
)

ax2.set_title(
    "B. The Value Ceiling",
    fontweight="bold"
)

ax2.set_xlabel("Hours released per month")
ax2.set_ylabel("3-year NPV (€)")

ax2.annotate(
    "Productive demand limit",
    xy=(
        available_demand_hours,
        npv_by_capacity.max()
    ),
    xytext=(11, npv_by_capacity.max() * 0.65),
    arrowprops={
        "arrowstyle": "->",
        "color": "#555555"
    },
    fontsize=10
)

fig_final.suptitle(
    "THE AUTOMATION VALUE TRAP",
    fontsize=19,
    fontweight="bold"
)

fig_final.savefig(
    MODEL_DIR / "automation_value_trap.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()


In [ ]:



# ============================================================
# STEP 34 — THE AUTOMATION VALUE TRAP
# Premium Dark Finance | LinkedIn 1080 x 1350
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, MultipleLocator
from pathlib import Path

# ------------------------------------------------------------
# 1. SCENARIO SETTINGS
# ------------------------------------------------------------

VISUAL_SCALE = 20
VISUAL_EXTRA_SETUP = 50
VISUAL_EXTRA_MAINTENANCE = 10

MODEL_DIR = PROJECT_DIR / "financial_model"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2. FINANCIAL MODEL
# ------------------------------------------------------------

annuity_factor = sum(
    1 / (1 + discount_rate) ** year
    for year in range(1, project_years + 1)
)

saved_minutes = max(
    manual_minutes - automated_minutes, 0
)

released_hours = VISUAL_SCALE * saved_minutes / 60

investment = (
    initial_investment
    + max(VISUAL_SCALE - 1, 0) * VISUAL_EXTRA_SETUP
)

maintenance = (
    annual_maintenance
    + max(VISUAL_SCALE - 1, 0) * VISUAL_EXTRA_MAINTENANCE
)

def calculate_npv(demand, contribution, capacity):
    productive = np.minimum(
        np.maximum(demand, 0),
        np.maximum(capacity, 0)
    )

    annual_benefit = productive * contribution * 12

    return (
        -investment
        + (annual_benefit - maintenance) * annuity_factor
    )

base_npv = float(
    calculate_npv(
        available_demand_hours,
        contribution_per_hour,
        released_hours
    )
)

productive_hours = min(
    released_hours,
    available_demand_hours
)

unused_hours = released_hours - productive_hours

# ------------------------------------------------------------
# 3. PREMIUM COLOR PALETTE
# ------------------------------------------------------------

BG = "#0B1220"          # Deep midnight navy
PANEL = "#111D30"       # Graph background
NAVY = "#0B1220"

WHITE = "#F8FAFC"
MUTED = "#A2B1C5"
GRID = "#314157"

GREEN = "#00D6A0"       # Electric emerald
GREEN_DARK = "#123F3B"  # Rich green region

ORANGE = "#FF8A3D"      # Vibrant amber-orange
ORANGE_DARK = "#4A3028" # Rich orange region

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "text.color": WHITE,
    "axes.labelcolor": WHITE,
    "axes.edgecolor": GRID,
    "xtick.color": MUTED,
    "ytick.color": MUTED,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "savefig.facecolor": BG
})

# ------------------------------------------------------------
# 4. PREPARE DATA
# ------------------------------------------------------------

demand_axis = np.linspace(0, 20, 400)
contribution_axis = np.linspace(0, 80, 400)

D, C = np.meshgrid(
    demand_axis,
    contribution_axis
)

NPV = calculate_npv(D, C, released_hours)

capacity_axis = np.linspace(0, 25, 400)

NPV_CAPACITY = calculate_npv(
    available_demand_hours,
    contribution_per_hour,
    capacity_axis
)

# ------------------------------------------------------------
# 5. CANVAS — 1080 x 1350
# ------------------------------------------------------------

fig = plt.figure(
    figsize=(10.8, 13.5),
    dpi=100,
    facecolor=BG
)

# Header
fig.text(
    0.075, 0.958,
    "AUTOMATION  /  INVESTMENT ANALYSIS",
    fontsize=10,
    color=GREEN,
    fontweight="bold"
)

fig.text(
    0.075, 0.914,
    "Automation Saves Time.",
    fontsize=27,
    color=WHITE,
    fontweight="bold"
)

fig.text(
    0.075, 0.874,
    "But Who Needs That Time?",
    fontsize=27,
    color=GREEN,
    fontweight="bold"
)

fig.text(
    0.075, 0.839,
    "The financial value of automation depends on what",
    fontsize=11,
    color=MUTED
)

fig.text(
    0.075, 0.820,
    "a business can actually do with the time it saves.",
    fontsize=11,
    color=MUTED
)

fig.add_artist(
    plt.Line2D(
        [0.075, 0.925],
        [0.795, 0.795],
        transform=fig.transFigure,
        color=GRID,
        linewidth=1.2
    )
)

# ------------------------------------------------------------
# 6. PANEL A — INVESTMENT DECISION MAP
# ------------------------------------------------------------

fig.text(
    0.075, 0.765,
    "01  /  WHEN DOES AUTOMATION PAY OFF?",
    fontsize=14,
    fontweight="bold",
    color=WHITE
)

ax1 = fig.add_axes([0.13, 0.493, 0.79, 0.235])
ax1.set_facecolor(PANEL)

decision = (NPV >= 0).astype(int)

ax1.contourf(
    D, C, decision,
    levels=[-0.5, 0.5, 1.5],
    colors=[ORANGE_DARK, GREEN_DARK],
    alpha=1
)

# Break-even boundary
if NPV.min() < 0 < NPV.max():
    ax1.contour(
        D, C, NPV,
        levels=[0],
        colors=[WHITE],
        linewidths=2.6
    )

# Capacity saturation boundary
if released_hours <= 20:
    ax1.axvline(
        released_hours,
        color=MUTED,
        linestyle=(0, (4, 4)),
        linewidth=1.4
    )

# Selected scenario
if (
    0 <= available_demand_hours <= 20
    and 0 <= contribution_per_hour <= 80
):
    ax1.scatter(
        available_demand_hours,
        contribution_per_hour,
        s=145,
        color=PANEL,
        edgecolor=WHITE,
        linewidth=2,
        zorder=10
    )

    ax1.scatter(
        available_demand_hours,
        contribution_per_hour,
        s=35,
        color=GREEN if base_npv >= 0 else ORANGE,
        zorder=11
    )

ax1.text(
    15.5, 65,
    "POSITIVE NPV",
    color=GREEN,
    fontsize=12,
    fontweight="bold",
    ha="center"
)

ax1.text(
    3.1, 12,
    "NEGATIVE NPV",
    color=ORANGE,
    fontsize=11,
    fontweight="bold",
    ha="center"
)

ax1.set_xlim(0, 20)
ax1.set_ylim(0, 80)

ax1.xaxis.set_major_locator(MultipleLocator(5))
ax1.yaxis.set_major_locator(MultipleLocator(20))

ax1.set_xlabel(
    "Productive demand (hours/month)",
    fontsize=11,
    labelpad=9
)

ax1.set_ylabel(
    "Incremental cash contribution (€/hour)",
    fontsize=10,
    labelpad=10
)

ax1.tick_params(labelsize=10)

ax1.grid(
    color=GRID,
    linewidth=0.7,
    alpha=0.45
)

fig.text(
    0.075, 0.452,
    "White line = break-even   |   Dot = selected scenario",
    fontsize=9.5,
    color=MUTED
)

# ------------------------------------------------------------
# 7. PANEL B — THE VALUE CEILING
# ------------------------------------------------------------

fig.text(
    0.075, 0.416,
    "02  /  THE VALUE CEILING",
    fontsize=14,
    fontweight="bold",
    color=WHITE
)

ax2 = fig.add_axes([0.13, 0.205, 0.79, 0.172])
ax2.set_facecolor(PANEL)

# Positive and negative regions
ax2.fill_between(
    capacity_axis,
    NPV_CAPACITY,
    0,
    where=NPV_CAPACITY >= 0,
    color=GREEN_DARK,
    interpolate=True
)

ax2.fill_between(
    capacity_axis,
    NPV_CAPACITY,
    0,
    where=NPV_CAPACITY < 0,
    color=ORANGE_DARK,
    interpolate=True
)

# Value curve
ax2.plot(
    capacity_axis,
    NPV_CAPACITY,
    color=GREEN,
    linewidth=3.5,
    solid_capstyle="round"
)

# Zero NPV
ax2.axhline(
    0,
    color=MUTED,
    linewidth=1.2
)

# Demand limit
ax2.axvline(
    available_demand_hours,
    color=ORANGE,
    linestyle=(0, (4, 4)),
    linewidth=2
)

# Selected scenario
if released_hours <= 25:
    ax2.scatter(
        released_hours,
        base_npv,
        s=110,
        color=GREEN if base_npv >= 0 else ORANGE,
        edgecolor=WHITE,
        linewidth=1.7,
        zorder=10
    )

# Demand limit label
ax2.annotate(
    "DEMAND LIMIT",
    xy=(
        available_demand_hours,
        np.max(NPV_CAPACITY)
    ),
    xytext=(
        available_demand_hours + 1,
        np.max(NPV_CAPACITY) * 0.70
    ),
    fontsize=9.5,
    fontweight="bold",
    color=ORANGE,
    arrowprops=dict(
        arrowstyle="-",
        color=ORANGE,
        linewidth=1.3
    )
)

ax2.set_xlim(0, 25)

ax2.set_xlabel(
    "Hours released per month",
    fontsize=11,
    labelpad=9
)

ax2.set_ylabel(
    "3-year NPV (€)",
    fontsize=11,
    labelpad=10
)

ax2.yaxis.set_major_formatter(
    FuncFormatter(
        lambda x, pos: f"€{x:,.0f}"
    )
)

ax2.tick_params(labelsize=10)

ax2.grid(
    axis="y",
    color=GRID,
    alpha=0.55,
    linewidth=0.8
)

# ------------------------------------------------------------
# 8. KEY INSIGHT
# ------------------------------------------------------------

fig.add_artist(
    plt.Line2D(
        [0.075, 0.925],
        [0.165, 0.165],
        transform=fig.transFigure,
        color=GRID,
        linewidth=1.2
    )
)

fig.text(
    0.075, 0.129,
    "TIME SAVED IS NOT CASH CREATED.",
    fontsize=19,
    fontweight="bold",
    color=WHITE
)

fig.text(
    0.075, 0.101,
    "Without productive demand, extra capacity has no",
    fontsize=11,
    color=MUTED
)

fig.text(
    0.075, 0.083,
    "additional cash benefit.",
    fontsize=11,
    color=MUTED
)

fig.text(
    0.075, 0.039,
    "PYTHON AUTOMATION  /  FINANCIAL MODELING  /  DECISION ANALYTICS",
    fontsize=9,
    fontweight="bold",
    color=GREEN
)

# ------------------------------------------------------------
# 9. EXPORT
# ------------------------------------------------------------

FINAL_PNG = (
    MODEL_DIR / "automation_value_trap_linkedin.png"
)

FINAL_PDF = (
    MODEL_DIR / "automation_value_trap_linkedin.pdf"
)

fig.savefig(
    FINAL_PNG,
    dpi=100,
    facecolor=BG
)

fig.savefig(
    FINAL_PDF,
    facecolor=BG
)

plt.show()
plt.close(fig)

print("\nFINAL VISUALIZATION")
print("-" * 45)
print("PNG:", FINAL_PNG)
print("PDF:", FINAL_PDF)
print(f"Released hours: {released_hours:.2f}/month")
print(f"Productive hours: {productive_hours:.2f}/month")
print(f"Unused hours: {unused_hours:.2f}/month")
print(f"3-year NPV: €{base_npv:,.2f}")


In [ ]:

from PIL import Image

with Image.open(FINAL_PNG) as img:
    print("Image size:", img.size)
    assert img.size == (1080, 1350)

print("LinkedIn image is ready.")
